In [1]:
import torch

from torch_openreml import MixedModelREML
from torch_openreml.covariance import DummyMatrix, ScalarMatrix

n, p = 10, 2

group = ["a", "b"] * 5
effect = torch.tensor([1.0, -1.0]).repeat(n // 2)

X = torch.randn(n, p)
y = X @ torch.tensor([0.5, -0.5]) + effect + 0.5 * torch.randn(n)

reml = MixedModelREML(
    genotype=(DummyMatrix(group), ScalarMatrix(group)),
)

names, free_params = reml.optimize(y, X, verbose=1)
names, free_params

[  1] ∥∇∥:       4.7998, ∥Δ∥: 2.1775, η: 0.50, ∥Δᶜ∥: 1.0888, accepted: True , log 𝓛:  -3.5994
[  2] ∥∇∥:       3.4880, ∥Δ∥: 0.2075, η: 1.00, ∥Δᶜ∥: 0.2075, accepted: True , log 𝓛:  -3.2312
[  3] ∥∇∥:       0.5666, ∥Δ∥: 0.0434, η: 1.00, ∥Δᶜ∥: 0.0434, accepted: True , log 𝓛:  -3.2186
[  4] ∥∇∥:       0.0234, ∥Δ∥: 0.0019, η: 1.00, ∥Δᶜ∥: 0.0019, accepted: True , log 𝓛:  -3.2186
[  5] ∥∇∥:       0.0000, ∥Δ∥: 0.0000, η: 1.00, ∥Δᶜ∥: 0.0000, accepted: True , log 𝓛:  -3.2186

[∇: score, Δ: 𝐉⁻¹∇, η: step length, Δᶜ: clip(𝛉 + ηΔ, lb, ub) - 𝛉, 𝓛: restricted likelihood]

✓ Converged at iteration 5, total time: 0.01s



(['G/sigma^2', 'R/sigma^2'], tensor([ 0.1894, -0.8235]))

In [2]:
history = reml.history
history["loglik"]

[tensor(-3.5994),
 tensor(-3.2312),
 tensor(-3.2186),
 tensor(-3.2186),
 tensor(-3.2186)]

In [3]:
torch.stack(history["loglik"])

tensor([-3.5994, -3.2312, -3.2186, -3.2186, -3.2186])

In [4]:
torch.stack(history["free_params"])

tensor([[ 0.2806, -1.0520],
        [ 0.1836, -0.8685],
        [ 0.1894, -0.8255],
        [ 0.1894, -0.8235],
        [ 0.1894, -0.8235]])

In [5]:
names, theta_hat = reml.get_free_params("last")
names, theta_hat

(['G/sigma^2', 'R/sigma^2'], tensor([ 0.1894, -0.8235]))

In [6]:
params = reml.build_params("last", out_format="dict")
params

{'G/sigma^2': tensor([1.4606]), 'R/sigma^2': tensor([0.1926])}

In [7]:
tree = reml.tree("last")
tree

{'/': tensor([[1.6532, 0.0000, 1.4606, 0.0000, 1.4606, 0.0000, 1.4606, 0.0000, 1.4606,
          0.0000],
         [0.0000, 1.6532, 0.0000, 1.4606, 0.0000, 1.4606, 0.0000, 1.4606, 0.0000,
          1.4606],
         [1.4606, 0.0000, 1.6532, 0.0000, 1.4606, 0.0000, 1.4606, 0.0000, 1.4606,
          0.0000],
         [0.0000, 1.4606, 0.0000, 1.6532, 0.0000, 1.4606, 0.0000, 1.4606, 0.0000,
          1.4606],
         [1.4606, 0.0000, 1.4606, 0.0000, 1.6532, 0.0000, 1.4606, 0.0000, 1.4606,
          0.0000],
         [0.0000, 1.4606, 0.0000, 1.4606, 0.0000, 1.6532, 0.0000, 1.4606, 0.0000,
          1.4606],
         [1.4606, 0.0000, 1.4606, 0.0000, 1.4606, 0.0000, 1.6532, 0.0000, 1.4606,
          0.0000],
         [0.0000, 1.4606, 0.0000, 1.4606, 0.0000, 1.4606, 0.0000, 1.6532, 0.0000,
          1.4606],
         [1.4606, 0.0000, 1.4606, 0.0000, 1.4606, 0.0000, 1.4606, 0.0000, 1.6532,
          0.0000],
         [0.0000, 1.4606, 0.0000, 1.4606, 0.0000, 1.4606, 0.0000, 1.4606, 0.0000,
    

In [8]:
beta_hat = reml.blue(y, X, theta_hat)
beta_hat

tensor([ 0.5495, -0.5096])

In [9]:
cov_beta = reml.cov_blue(X, theta_hat)
cov_beta

tensor([[0.0250, 0.0055],
        [0.0055, 0.0215]])

In [10]:
se_beta = torch.sqrt(reml.var_blue(X, theta_hat))
se_beta

tensor([0.1581, 0.1467])

In [11]:
b_hat = reml.blup()
b_hat

tensor([ 1.0284, -1.3340])

In [12]:
b_cov = reml.cov_blup()
b_cov

tensor([[ 1.4212, -0.0036],
        [-0.0036,  1.4159]])

In [13]:
b_var = reml.var_blup()
b_var

tensor([1.4212, 1.4159])

In [14]:
b_by_term = reml.blup_per_term()
b_by_term

{'genotype': tensor([ 1.0284, -1.3340])}

In [15]:
b_cov_by_term = reml.cov_blup_per_term()
b_cov_by_term

{'genotype': tensor([[ 1.4212, -0.0036],
         [-0.0036,  1.4159]])}

In [16]:
b_var_by_term = reml.var_blup_per_term()
b_var_by_term

{'genotype': tensor([1.4212, 1.4159])}

In [17]:
y_marginal = reml.marginal_predict()
y_marginal

tensor([ 0.3317,  0.8644, -0.7591, -1.2185, -0.0440, -0.1003,  1.0306,  1.5697,
         0.2316,  0.1966])

In [18]:
y_conditional = reml.predict()
y_conditional

tensor([ 1.3600, -0.4697,  0.2692, -2.5525,  0.9844, -1.4343,  2.0590,  0.2357,
         1.2600, -1.1375])

In [19]:
y_new = reml.predict_new(
    X[:2],
    {"genotype": torch.tensor([[1.0, 0.0], [0.0, 1.0]])},
)
y_new

tensor([ 1.3600, -0.4697])

In [20]:
z_new = DummyMatrix(group).transform(["a", "b"])

y_new = reml.predict_new(
    X[:2],
    z_new,
)
y_new

tensor([ 1.3600, -0.4697])

In [21]:
e_marginal = reml.marginal_residual(y, X, theta_hat)
e_marginal

tensor([ 1.6104, -1.2705,  0.6012, -0.9624,  1.4019, -1.7037,  0.8124, -1.2095,
         0.8516, -1.6999])

In [22]:
e_conditional = reml.residual(y, X, theta_hat)
e_conditional

tensor([ 0.5820,  0.0635, -0.4272,  0.3716,  0.3735, -0.3697, -0.2160,  0.1245,
        -0.1768, -0.3658])

In [23]:
c11, c12, c21, c22 = reml.c_inverse_blocks(X, theta_hat)
c11, c12, c21, c22

(tensor([[0.0250, 0.0055],
         [0.0055, 0.0215]]),
 tensor([[-0.0068, -0.0133],
         [-0.0013, -0.0041]]),
 tensor([[-0.0068, -0.0013],
         [-0.0133, -0.0041]]),
 tensor([[0.0394, 0.0036],
         [0.0036, 0.0446]]))

In [24]:
loglik_value, loglik_method = reml.loglik(y, X, theta_hat)
loglik_value, loglik_method

(tensor(-3.2186), 'cform')

In [25]:
reml.is_converged()

True